# GPU-Accelerated Data Analysis with NVIDIA RAPIDS
### Case study: New York City yellow taxi trips (NYC TLC)

This notebook runs the **same analysis twice**, on CPU (pandas, scikit-learn) and on GPU (**cuDF**, **cuML**), and measures the speed-up at each step:

1. Load the data (Parquet)
2. Clean it and create features
3. Business aggregations (revenue, trips and tips by hour and zone)
4. Join with the taxi zone table
5. Customer segmentation with K-Means
6. Tip prediction with a Random Forest
7. Benchmark summary

**Runtime:** Google Colab → *Runtime → Change runtime type → GPU* (T4 or better).

**Data:** [NYC TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page), public monthly Parquet files.

## 1. Check the GPU

In [ ]:
!nvidia-smi

## 2. Install RAPIDS
cuDF is preinstalled on Colab. The official RAPIDS script below installs the rest of the suite (cuML, etc.). It takes a few minutes.
Source: [RAPIDS on Google Colab](https://docs.rapids.ai/deployment/stable/platforms/colab/).

> If Colab asks you to restart the session after installing, restart it and continue from the next cell.

In [ ]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-utils/colab/pip-install.py

## 3. Imports and configuration

In [ ]:
import os, time, statistics
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import cudf
import cuml
from cuml.cluster import KMeans as cuKMeans
from cuml.ensemble import RandomForestRegressor as cuRF

import sklearn
from sklearn.cluster import KMeans as skKMeans
from sklearn.ensemble import RandomForestRegressor as skRF
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__)
print("cuDF  ", cudf.__version__, "| cuML", cuml.__version__)

# ---- Settings ---------------------------------------------------------------
MONTHS = ["2024-01"]      # add more months (e.g. "2024-02", "2024-03") to increase the volume
N_ML = 500_000            # rows used for the Random Forest (keeps CPU training time reasonable)
REPEATS = 3               # each timing is the median of REPEATS runs
BASE_URL = "https://d37ci6vzurychx.cloudfront.net"

os.makedirs("data", exist_ok=True)
os.makedirs("results", exist_ok=True)
os.makedirs("docs", exist_ok=True)

## 4. Download the data

In [ ]:
import urllib.request

files = []
for m in MONTHS:
    path = f"data/yellow_tripdata_{m}.parquet"
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{BASE_URL}/trip-data/yellow_tripdata_{m}.parquet", path)
    files.append(path)

ZONES = "data/taxi_zone_lookup.csv"
if not os.path.exists(ZONES):
    urllib.request.urlretrieve(f"{BASE_URL}/misc/taxi_zone_lookup.csv", ZONES)

for f in files + [ZONES]:
    print(f, f"{os.path.getsize(f) / 1e6:.1f} MB")

## 5. Benchmark helper
Each step runs the CPU version and the GPU version `REPEATS` times and keeps the **median** time. The first GPU run includes some warm-up, which is why the median is used.

In [ ]:
results = []

def bench(step, cpu_fn, gpu_fn, repeats=REPEATS):
    def timed(fn):
        times, out = [], None
        for _ in range(repeats):
            t0 = time.perf_counter()
            out = fn()
            times.append(time.perf_counter() - t0)
        return out, statistics.median(times)

    cpu_out, t_cpu = timed(cpu_fn)
    gpu_out, t_gpu = timed(gpu_fn)
    results.append({"step": step, "cpu_s": t_cpu, "gpu_s": t_gpu, "speedup": t_cpu / t_gpu})
    print(f"{step:<30} CPU {t_cpu:8.3f} s | GPU {t_gpu:8.3f} s | x{t_cpu / t_gpu:6.1f}")
    return cpu_out, gpu_out

## 6. Load the data (pandas vs cuDF)

In [ ]:
COLS = ["tpep_pickup_datetime", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
        "PULocationID", "DOLocationID", "payment_type", "fare_amount", "tip_amount", "total_amount"]

pdf_raw, gdf_raw = bench(
    "Load Parquet",
    lambda: pd.concat([pd.read_parquet(f, columns=COLS) for f in files], ignore_index=True),
    lambda: cudf.concat([cudf.read_parquet(f, columns=COLS) for f in files], ignore_index=True),
)
print(f"{len(pdf_raw):,} rows")
pdf_raw.head()

## 7. Clean the data and create features
The **same function** works for pandas and cuDF, because cuDF follows the pandas API.

Rules: realistic distance, fare, duration and speed, and pick-up dates inside the selected months.

In [ ]:
YEARS = sorted({int(m[:4]) for m in MONTHS})

def clean(df):
    df = df.copy()
    pick = df["tpep_pickup_datetime"].astype("datetime64[ns]")
    drop = df["tpep_dropoff_datetime"].astype("datetime64[ns]")
    df["duration_min"] = (drop.astype("int64") - pick.astype("int64")) / 6e10
    df["hour"] = pick.dt.hour
    df["dow"] = pick.dt.dayofweek          # 0 = Monday
    df["year"] = pick.dt.year
    df = df[
        (df["trip_distance"] > 0.1) & (df["trip_distance"] < 100)
        & (df["fare_amount"] > 0) & (df["fare_amount"] < 500)
        & (df["tip_amount"] >= 0) & (df["total_amount"] > 0)
        & (df["duration_min"] >= 1) & (df["duration_min"] <= 180)
        & (df["year"].isin(YEARS))
    ]
    df = df.assign(
        speed_mph=df["trip_distance"] / (df["duration_min"] / 60),
        tip_pct=df["tip_amount"] / df["fare_amount"] * 100,
    )
    return df[df["speed_mph"] < 80]

pdf, gdf = bench("Clean + features", lambda: clean(pdf_raw), lambda: clean(gdf_raw))
print(f"Kept {len(pdf):,} of {len(pdf_raw):,} rows ({len(pdf) / len(pdf_raw):.1%})")

## 8. Business aggregations

In [ ]:
def by_hour(df):
    out = df.groupby("hour").agg({"total_amount": ["count", "sum"], "tip_pct": "mean"})
    out.columns = ["trips", "revenue", "avg_tip_pct"]
    return out.sort_index()

def by_zone(df):
    out = df.groupby("PULocationID").agg({"total_amount": ["count", "sum"], "trip_distance": "mean"})
    out.columns = ["trips", "revenue", "avg_distance"]
    return out.sort_values("revenue", ascending=False)

def by_payment(df):
    out = df.groupby("payment_type").agg({"tip_amount": "mean", "tip_pct": "mean", "total_amount": "count"})
    out.columns = ["avg_tip", "avg_tip_pct", "trips"]
    return out.sort_index()

hour_cpu, hour_gpu = bench("Group by hour", lambda: by_hour(pdf), lambda: by_hour(gdf))
zone_cpu, zone_gpu = bench("Group by pick-up zone", lambda: by_zone(pdf), lambda: by_zone(gdf))
pay_cpu, pay_gpu = bench("Group by payment type", lambda: by_payment(pdf), lambda: by_payment(gdf))

hour_gpu.to_pandas().head(24)

**Payment types:** 1 = credit card, 2 = cash (see the TLC data dictionary).
The TLC data dictionary notes that the tip field only records credit card tips, so cash tips appear as 0. Analyse tips on card payments only.

In [ ]:
pay_gpu.to_pandas()

## 9. Join with the taxi zone table

In [ ]:
zones_cpu = pd.read_csv(ZONES)
zones_gpu = cudf.read_csv(ZONES)

def top_zones(df, zones):
    z = by_zone(df).reset_index().merge(zones, left_on="PULocationID", right_on="LocationID", how="left")
    return z.sort_values("revenue", ascending=False).head(10)[["Borough", "Zone", "trips", "revenue", "avg_distance"]]

top_cpu, top_gpu = bench("Join + top 10 zones", lambda: top_zones(pdf, zones_cpu), lambda: top_zones(gdf, zones_gpu))
top_gpu.to_pandas()

## 10. Visualise the business insights

In [ ]:
h = hour_gpu.to_pandas()
t = top_gpu.to_pandas()

fig, axes = plt.subplots(1, 3, figsize=(18, 4.5))
axes[0].bar(h.index, h["trips"] / 1e3, color="#76B900")
axes[0].set(title="Trips by pick-up hour", xlabel="Hour", ylabel="Trips (thousands)")
axes[1].plot(h.index, h["revenue"] / 1e6, marker="o", color="#1F2937")
axes[1].set(title="Revenue by pick-up hour", xlabel="Hour", ylabel="Revenue (million $)")
axes[2].barh(t["Zone"][::-1], t["revenue"][::-1] / 1e6, color="#76B900")
axes[2].set(title="Top 10 pick-up zones by revenue", xlabel="Revenue (million $)")
plt.tight_layout()
plt.savefig("docs/business_insights.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. Segmentation with K-Means (scikit-learn vs cuML)
Features are standardised the same way on CPU and GPU. Both models use 5 clusters, one initialisation and the same random seed.

In [ ]:
SEG_FEATS = ["trip_distance", "duration_min", "fare_amount", "hour"]

def scaled(df):
    X = df[SEG_FEATS].astype("float32")
    return (X - X.mean()) / X.std()

Xc, Xg = scaled(pdf), scaled(gdf)
cuKMeans(n_clusters=5, n_init=1, random_state=42).fit(Xg.head(10_000))   # GPU warm-up (not timed)

sk_km, cu_km = bench(
    "K-Means (5 clusters)",
    lambda: skKMeans(n_clusters=5, n_init=1, random_state=42).fit(Xc),
    lambda: cuKMeans(n_clusters=5, n_init=1, random_state=42).fit(Xg),
    repeats=1,
)

profile = (pdf[SEG_FEATS + ["tip_pct"]]
           .assign(cluster=sk_km.labels_)
           .groupby("cluster")
           .agg(["mean"]).round(1))
profile.columns = [c[0] for c in profile.columns]
profile["share_%"] = (pd.Series(sk_km.labels_).value_counts(normalize=True).sort_index() * 100).round(1)
profile

**Interpret the clusters** in business terms (for example: short daytime city trips, airport trips, long night trips) and write your conclusions here.

## 12. Tip prediction with a Random Forest (scikit-learn vs cuML)
Card payments only (cash tips are not recorded). The same sample and the same train/test split are used on CPU and GPU, so the scores are comparable.

In [ ]:
ML_FEATS = ["trip_distance", "duration_min", "fare_amount", "hour", "dow", "PULocationID", "DOLocationID", "passenger_count"]

sample = (pdf[pdf["payment_type"] == 1]
          .dropna(subset=ML_FEATS + ["tip_amount"])
          .sample(n=min(N_ML, int((pdf["payment_type"] == 1).sum())), random_state=42))
X = sample[ML_FEATS].astype("float32")
y = sample["tip_amount"].astype("float32")
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

gX_tr, gX_te = cudf.from_pandas(X_tr), cudf.from_pandas(X_te)
gy_tr = cudf.from_pandas(y_tr)

RF_PARAMS = dict(n_estimators=50, max_depth=10, random_state=42)
sk_rf, cu_rf = bench(
    "Random Forest (train)",
    lambda: skRF(n_jobs=-1, **RF_PARAMS).fit(X_tr, y_tr),
    lambda: cuRF(**RF_PARAMS).fit(gX_tr, gy_tr),
    repeats=1,
)

r2_cpu = r2_score(y_te, sk_rf.predict(X_te))
def to_numpy(a):   # cuML may return a cuDF Series or a CuPy array depending on its output settings
    return a.to_numpy() if hasattr(a, "to_numpy") else (a.get() if hasattr(a, "get") else np.asarray(a))

r2_gpu = r2_score(y_te, to_numpy(cu_rf.predict(gX_te)))
print(f"R² on test set: scikit-learn {r2_cpu:.3f} | cuML {r2_gpu:.3f}")

## 13. Benchmark summary

In [ ]:
bench_df = pd.DataFrame(results)
bench_df["speedup"] = bench_df["speedup"].round(1)
bench_df.to_csv("results/benchmarks.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(bench_df["step"][::-1], bench_df["speedup"][::-1], color="#76B900")
for i, v in enumerate(bench_df["speedup"][::-1]):
    ax.text(v, i, f" x{v}", va="center")
ax.set(title=f"GPU speed-up vs CPU ({len(pdf_raw):,} rows)", xlabel="Speed-up (CPU time / GPU time)")
plt.tight_layout()
plt.savefig("docs/benchmark_chart.png", dpi=150, bbox_inches="tight")
plt.show()

bench_df.round(3)

## 14. Environment (for reproducibility)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!lscpu | grep "Model name"
!nproc
print("Rows analysed:", f"{len(pdf_raw):,}", "| Months:", MONTHS)
print("pandas", pd.__version__, "| scikit-learn", sklearn.__version__, "| cuDF", cudf.__version__, "| cuML", cuml.__version__)

## 15. Bonus: accelerate existing pandas code with `cudf.pandas`
`cudf.pandas` runs normal pandas code on the GPU without changing it. It replaces pandas for the whole session, so try it in a **new notebook** (not in this one, where CPU pandas is used as the baseline):

```python
%load_ext cudf.pandas      # first cell, before importing pandas
import pandas as pd
df = pd.read_parquet("data/yellow_tripdata_2024-01.parquet")
df.groupby("PULocationID")["total_amount"].sum().sort_values(ascending=False).head()
```

## 16. Conclusions
Write your findings here, using only the numbers measured above:

- **Performance:** which steps gained the most from the GPU, and why (large aggregations, joins, ML training).
- **Model quality:** cuML and scikit-learn R² scores are similar, so the speed-up does not reduce accuracy.
- **Business insights:** busiest and most profitable hours, top zones, tipping behaviour, customer segments.
- **Limits:** results depend on the GPU/CPU used and on the data volume; try more months to see the gap grow.